# Session 2 — Feature Engineering

In session 1 the data was ready to use: every column was a number, and nothing was missing. Real
datasets are not like that. Values are missing. Most columns hold text, not numbers. Quantities
come in very different scales. And a few rows hold extreme values that distort everything around
them.

**Feature engineering** is the work of turning raw columns into features that a model can use.
This session is a practical guide to it. We explain each technique, show it on a small example you
can read at a glance, and then apply it to a real dataset.

We do **not** fit a model in this session. Whether a technique improves predictions is a question
about measurement, and measurement is session 3. Mixing the two would move your attention away
from the preparation itself, which is the subject here.

## Learning objectives

After this session you should be able to:

1. Say what **type** each column is, and which techniques that type allows.
2. Handle **missing values** based on what the absence means.
3. Find **outliers** and choose between clipping, transforming and robust scaling.
4. Encode **categorical** variables, and tell ordered ones from unordered ones.
5. **Scale** numeric variables, and say which method fits which situation.
6. Build **new features** from the columns you already have.
7. Run a **diagnostic** on a new dataset that tells you which technique each column needs.
8. Put everything into one `scikit-learn` pipeline, and explain why doing the same work by hand in
   `pandas` causes data leakage.

The cell below imports everything the notebook needs. We call
`set_output(transform="pandas")` on most transformers. This makes them return a DataFrame instead
of a plain array, so we can read what each step did.

In [1]:
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (FunctionTransformer, MinMaxScaler, OneHotEncoder,
                                   OrdinalEncoder, RobustScaler, StandardScaler)

pd.set_option('display.max_columns', 50)
RANDOM_STATE = 42

# Part 1 — Types of features

Every technique in this session applies to one kind of column. So we start by naming the kinds.

**Numerical features** hold quantities. They come in two forms. **Continuous** values can take any
value in a range, such as a floor area or a price. **Discrete** values are counts, such as the
number of bedrooms. Arithmetic on them has meaning: twice the area really is twice as much space.

**Categorical features** hold a label from a fixed set. They come in three forms:

- **Nominal** — the values have no order. A neighbourhood name is nominal. "Gilbert" is not
  greater than "Edwards".
- **Ordinal** — the values have a real order. A quality grade from Poor to Fair to Typical to Good
  to Excellent is ordinal.
- **Binary** — the column has exactly two values. This is a special case that needs only one 0/1
  column.

**Text** and **dates** are two more types, and each has its own techniques. Text belongs to the
planned GenAI course. Dates and time series belong to the planned time-series course. We do not
cover either one here.

The most important distinction is **nominal against ordinal**. You cannot see it in the data type,
because both arrive as text. You only know which one you have if you know what the values mean.

The cell below builds a toy table of six house sales, small enough to read in
full. It contains one of every problem this session covers.

Two of its columns use **short codes** rather than words. These codes come from the real dataset we
use in Part 4, so it is worth learning them now. They appear throughout the session:

| Column | Code | Meaning |
|---|---|---|
| `quality` | `Po` | Poor |
| `quality` | `Fa` | Fair |
| `quality` | `TA` | Typical (average) |
| `quality` | `Gd` | Good |
| `quality` | `Ex` | Excellent |
| `garage` | `Attchd` | garage attached to the house |
| `garage` | `Detchd` | garage separate from the house |

`TA` is the code you cannot guess. It means **Typical**, the middle grade, and it is the most
common value in the real data.

In [2]:
homes = pd.DataFrame({
    "lot_area":     [8450, 9600, 11250, 9550, 92000, 7420],
    "living_area":  [1710, 1262, 1786, 1717, 2198, 1362],
    "frontage":     [65.0, 80.0, np.nan, 60.0, 84.0, np.nan],
    "bedrooms":     [3, 3, 3, 4, 4, 2],
    "quality":      ["Gd", "TA", "Gd", "Ex", "TA", "Fa"],
    "neighborhood": ["Gilbert", "Edwards", "Gilbert", "Crawfor", "NAmes", "Edwards"],
    "garage":       ["Attchd", "Attchd", np.nan, "Detchd", "Attchd", np.nan],
})
homes

,lot_area,living_area,frontage,bedrooms,quality,neighborhood,garage
0,8450,1710,65.0,3,Gd,Gilbert,Attchd
1,9600,1262,80.0,3,TA,Edwards,Attchd
2,11250,1786,NaN,3,Gd,Gilbert,NaN
3,9550,1717,60.0,4,Ex,Crawfor,Detchd
4,92000,2198,84.0,4,TA,NAmes,Attchd
5,7420,1362,NaN,2,Fa,Edwards,NaN


Look at each column in turn. Part 2 works through exactly these problems:

| Column | Type | The problem |
|---|---|---|
| `lot_area` | numeric, continuous | row 4 holds **92,000** while the others are near 9,000. This is an outlier. |
| `living_area` | numeric, continuous | no problem |
| `frontage` | numeric, continuous | **two values are missing** |
| `bedrooms` | numeric, discrete | a count, on a very different scale from the areas |
| `quality` | categorical, **ordinal** | `Fa` < `TA` < `Gd` < `Ex`, that is Fair < Typical < Good < Excellent. The order has meaning. |
| `neighborhood` | categorical, **nominal** | four labels with no order |
| `garage` | categorical, nominal | **two missing** — and the absence means *no garage* |

Seven columns, five different problems.

# Part 2 — The techniques

## 2.0 How a scikit-learn transformer works

Every technique below is applied by an object called a **transformer**. They all work the same way,
so it is worth seeing the pattern once before we use it many times.

A transformer has two steps:

1. **`fit`** — look at the data and *learn* the values it needs. A median. A mean. The list of
   categories that exist.
2. **`transform`** — use those learned values to change data. You can transform any data, including
   rows the transformer has never seen.

The two steps are separate on purpose, and that separation is the subject of Part 5.

Let us make this concrete with `StandardScaler`, the transformer used in the cells
below.

**What it does.** It rewrites a numeric column so that the column ends up with a mean of 0 and a
standard deviation of 1. This is called **standardisation**. Section 2.1 returns to it and compares
it against the alternatives.

**What it has to learn.** To rewrite a single value, the transformer applies this formula:

$$x' = \frac{x - \mu}{\sigma}$$

where:

- $x$ — the original value
- $\mu$ — the mean of the column
- $\sigma$ — the standard deviation of the column; $\sigma > 0$
- $x'$ — the standardised value

Now look at what that formula needs. Neither $\mu$ nor $\sigma$ is available in the row being
transformed: both of them describe the **whole column**. They have to be computed from data before
any value can be rewritten, and **`fit` is the step that computes them**.

**Where it keeps them.** Everything a transformer learns during `fit` is stored on the object, in an
attribute whose name **ends with an underscore**. `StandardScaler` stores $\mu$ as `.mean_` and
$\sigma$ as `.scale_`. This rule holds across the whole library: a trailing underscore marks a
value that was learned from data, not a setting you chose.

```mermaid
flowchart LR
    A["raw column<br/>living_area"] -- "fit()" --> B["StandardScaler<br/>stores mean_ and scale_"]
    B -- "transform()" --> C["standardised column<br/>mean 0, spread 1"]
```

The next three cells run this in slow motion. The first one simply describes the column as it
arrives, so we have something to compare against.

In [3]:
print("living_area, as it arrives")
print(f"  mean               : {np.mean(homes['living_area']):>10,.2f}")
print(f"  standard deviation : {np.std(homes['living_area']):>10,.2f}")

homes[["living_area"]]

living_area, as it arrives
  mean               :   1,672.50
  standard deviation :     304.78


,living_area
0,1710
1,1262
2,1786
3,1717
4,2198
5,1362


The column has a mean of **1,672.50** and a standard deviation of
**304.78**. Those are the two numbers the formula needs, and we have just computed them ourselves.

The cell below creates the transformer and calls `fit`, so that it computes the same two numbers and
stores them. Compare what it prints against the two values above.

In [4]:
scaler = StandardScaler().set_output(transform="pandas")  # transformer is instantiated

scaler.fit(homes[["living_area"]])  # transformer learns from data

print(f"learned mean   (.mean_)  : {scaler.mean_[0]:>10,.2f}")
print(f"learned spread (.scale_) : {scaler.scale_[0]:>10,.2f}")

learned mean   (.mean_)  :   1,672.50
learned spread (.scale_) :     304.78


The scaler stored exactly the numbers we computed ourselves: `mean_`
is 1,672.50 and `scale_` is 304.78. Nothing was invented — `fit` measured the column.

The cell below calls `transform`, which applies the formula to every row, and then measures the
result the same way we measured the original.

In [5]:
standardised = scaler.transform(homes[["living_area"]])

print("living_area, after standardising")
print(f"  mean               : {np.mean(standardised['living_area']):>10,.2f}")
print(f"  standard deviation : {np.std(standardised['living_area']):>10,.2f}")

standardised.round(3)

living_area, after standardising
  mean               :       0.00
  standard deviation :       1.00


,living_area
0,0.123
1,-1.347
2,0.372
3,0.146
4,1.724
5,-1.019


The standardised column has a mean of **0.00** and a standard deviation of
**1.00**, which is exactly what standardisation promises.

Notice that the shape of the column did not change. Every value was rewritten with the same two
numbers, so the largest house is still the largest and the gaps between houses keep their
proportions. Standardisation changes the units, not the pattern.

That is the whole idea, and every technique in this session follows it: **`fit` measures something
and stores it; `transform` applies what was stored.**

You will usually see `fit_transform`, which does both steps in one call on the same data. It is a
shortcut, and it is the right thing to use on training data. It is the wrong thing to use on test
data, and Part 5 explains why.

> **In `scikit-learn`**
>
> | Code | What it does |
> |---|---|
> | `.fit(X)` | learn from `X`; returns the transformer itself |
> | `.transform(X)` | apply what was learned to `X`; returns the changed data |
> | `.fit_transform(X)` | both, on the same data |
> | `.set_output(transform="pandas")` | return a DataFrame with column names instead of a plain array |
> | a trailing `_`, as in `.mean_` | marks a value that was learned during `fit` |
>
> We measure with `np.mean()` and `np.std()` rather than the `pandas` methods of the same name,
> because `np.std()` computes spread the same way `StandardScaler` does. The `pandas` version uses a
> different convention and would report a different number.

## 2.1 Feature scaling

Numeric columns often use very different scales. In our table, `lot_area` reaches 92,000 while
`bedrooms` runs from 2 to 4. Some methods compare or combine features across columns: they measure
distances between rows, or they penalise large coefficients. In those methods the column with large
values will dominate, only because of its units.

There are two standard methods.

**Normalisation (min-max scaling)** maps every value into the range $[0, 1]$:

$$x' = \frac{x - \min(x)}{\max(x) - \min(x)}$$

where:

- $x$ — the original value
- $\min(x)$, $\max(x)$ — the smallest and largest values **in the training data**
- $x'$ — the scaled value. It is in $[0, 1]$ for the training data, and may fall outside that range
  for new data.

**Standardisation (z-score scaling)** gives the column a mean of 0 and a standard deviation of 1:

$$x' = \frac{x - \mu}{\sigma}$$

where:

- $\mu$ — the mean of the column in the training data
- $\sigma$ — the standard deviation of the column in the training data; $\sigma > 0$
- $x'$ — the scaled value. It is usually between −3 and 3, but it has no fixed limits.

> **In `scikit-learn`**
>
> | Code | What it does |
> |---|---|
> | `MinMaxScaler()` | maps each column into $[0, 1]$ using its smallest and largest value |
> | `StandardScaler()` | gives each column mean 0 and spread 1; stores `.mean_` and `.scale_` |
> | `RobustScaler()` | uses the median and IQR instead, for columns that contain outliers |

The cell below applies the first two methods to `lot_area` and `bedrooms` together.

In [6]:
homes[["lot_area", "bedrooms"]]

,lot_area,bedrooms
0,8450,3
1,9600,3
2,11250,3
3,9550,4
4,92000,4
5,7420,2


In [7]:
minmax_scaler = MinMaxScaler().set_output(transform="pandas")
minmax_scaler.fit_transform(homes[["lot_area", "bedrooms"]])

,lot_area,bedrooms
0,0.012178,0.5
1,0.025774,0.5
2,0.045283,0.5
3,0.025183,1.0
4,1.000000,1.0
5,0.000000,0.0


In [8]:
std_scaler = StandardScaler().set_output(transform="pandas")
std_scaler.fit_transform(homes[["lot_area", "bedrooms"]])

,lot_area,bedrooms
0,-0.472946,-0.242536
1,-0.435681,-0.242536
2,-0.382213,-0.242536
3,-0.437301,1.212678
4,2.234463,1.212678
5,-0.506323,-1.697749


Both methods put the two columns on a comparable scale. But they fail in
different ways when an outlier is present.

`MinMaxScaler` maps 92,000 to 1.0 and pushes the five ordinary houses into the range **0.0 to
0.045**. The range is defined by the smallest and largest values, so one extreme value takes almost
the whole interval. `StandardScaler` spreads them a little more, but they are still close to −0.45.

So the real choice is between three methods: min-max, standard, and a third one called **robust
scaling**. We cannot explain the third one yet, because it is built on a measure of spread that we
define in section 2.5, when we deal with outliers. Which method is right depends on the shape of
the column, and Part 3 shows how to check that shape.

> **In `scikit-learn`**
>
> | Code | What it does |
> |---|---|
> | `MinMaxScaler()` | maps each column into $[0, 1]$ using its smallest and largest value |
> | `StandardScaler()` | gives each column mean 0 and spread 1; stores `.mean_` and `.scale_` |

## 2.2 Missing values

Most models cannot accept a missing value. Every gap must be removed or filled.

**Deletion** removes the rows, or the columns, that contain gaps. Use it only when very little data
is affected. It also throws away everything else those rows contained.

**Imputation** fills the gap with a substitute value computed from the data you have. The
substitute can be a mean, a median, the most common category, or a fixed value you choose.

The right substitute depends on **what the absence means**. There are three common cases:

- **The value is unknown.** It exists, but nobody recorded it. Use a plausible substitute: the
  median for a skewed column, the mean for a symmetric one.
- **The value does not apply.** There is no value because the thing does not exist. `garage` is
  missing for houses that have no garage. Here the right answer is a **new category** that says so.
- **The value is zero.** The quantity is really nothing. Fill it with 0, not with a median.

> 🛠️ **In `scikit-learn`**, `SimpleImputer()` is the work horse for data imputation.
>
> | Code | What it does |
> |---|---|
> | `strategy="XXXXX"` | fills gaps with a given strategy: `"median"`, `"mean"`, `"most_frequent"` and `"constant"`. |
> | `fill_value="NoGarage"` | the value written into the gaps when the strategy is `"constant"` |
> | `add_indicator=True` | adds an extra 0/1 column that marks where values were missing |
> | `.statistics_` | the fill values the imputer learned during `fit` |

The cell below fills `frontage` in two ways. It prints the value that each strategy learned.

In [9]:
print("original values:")
print(homes["frontage"].to_list())

original values:
[65.0, 80.0, nan, 60.0, 84.0, nan]


In [10]:
imputer = SimpleImputer(strategy="median")  # Try also strategy="mean"
transformed_col = imputer.fit_transform(homes[["frontage"]])

print("Tranformed values:")
print(transformed_col.ravel())

Tranformed values:
[65.  80.  72.5 60.  84.  72.5]


The median is **72.50** and the mean is **72.25**. They are close here because
this column has no extreme values. In a skewed column they would be far apart, and the median
would be the safer choice.

For a categorical column you cannot compute a mean. You choose between the most common category
and a category you define yourself. The next cell shows both options on `garage`. Remember that
two of these houses have no garage at all.

In [11]:
print("original values:")
print(homes["garage"].to_list())

original values:
['Attchd', 'Attchd', nan, 'Detchd', 'Attchd', nan]


In [12]:
constant_imputer = SimpleImputer(strategy="constant", fill_value="NoGarage")
filled = constant_imputer.fit_transform(homes[["garage"]])

print("Tranformed values:")
print(filled.ravel())

Tranformed values:
['Attchd' 'Attchd' 'NoGarage' 'Detchd' 'Attchd' 'NoGarage']


In [13]:
frequent_imputer = SimpleImputer(strategy="most_frequent")
filled = frequent_imputer.fit_transform(homes[["garage"]])

print("Tranformed values:")
print(filled.ravel())

Tranformed values:
['Attchd' 'Attchd' 'Attchd' 'Detchd' 'Attchd' 'Attchd']


`most_frequent` writes **`Attchd`** into both gaps. This says that two houses
have an attached garage, when in fact they have no garage. `constant` writes **`NoGarage`**, which
is what the data really means.

This is the most important decision in this section, and it is not a technical one. You need to
know what the blank means.

There is also a way to avoid the choice. You can keep **both** pieces of information: the
substitute value, and a flag that records that the value was missing. The cell below adds that
flag.

In [14]:
imputer = SimpleImputer(strategy="median", add_indicator=True).set_output(transform="pandas")
imputer.fit_transform(homes[["frontage"]])

,frontage,missingindicator_frontage
0,65.0,0.0
1,80.0,0.0
2,72.5,1.0
3,60.0,0.0
4,84.0,0.0
5,72.5,1.0


The result has two columns. The first is `frontage` with the gaps filled. The
second is `missingindicator_frontage`, which holds 1 where a value was missing and 0 elsewhere.

The model can now use the substitute value *and* learn whether "we did not know this" is useful
for prediction. It often is, because data goes missing for reasons.

## 2.3 Encoding categorical variables

A model works with numbers, so categories must become numbers. The right encoding depends on
whether the categories have an order.

### 2.3.1. Nominal variables

**One-hot encoding** is for **nominal** variables:

```mermaid
flowchart LR
    subgraph Before
        A[":::markdown
| Color   |
|---------|
| Red     |
| Blue    |
| Green   |
| Red     |
| Green   |
"]
    end

    subgraph After
        B[":::markdown
| Red | Blue | Green |
|-----|------|-------|
|  1  |  0   |   0   |
|  0  |  1   |   0   |
|  0  |  0   |   1   |
|  1  |  0   |   0   |
|  0  |  0   |   1   |
"]
    end

    A --> B
```

Each category becomes its own column. The column
holds 1 when the row belongs to that category and 0 otherwise. This assumes nothing about how the
categories relate to each other. The cost is one column per category.

The cell below applies one-hot encoding to `neighborhood`.

In [15]:
encoder = OneHotEncoder(sparse_output=False).set_output(transform="pandas")
encoder.fit_transform(homes[["neighborhood"]])

,neighborhood_Crawfor,neighborhood_Edwards,neighborhood_Gilbert,neighborhood_NAmes
0,0.0,0.0,1.0,0.0
1,0.0,1.0,0.0,0.0
2,0.0,0.0,1.0,0.0
3,1.0,0.0,0.0,0.0
4,0.0,0.0,0.0,1.0
5,0.0,1.0,0.0,0.0


Four categories became four columns. Each row holds exactly one 1. The
encoding says nothing about how the neighbourhoods relate to each other. This is correct, because
they have no order.

> **In `scikit-learn`**
>
> | Code | What it does |
> |---|---|
> | `OneHotEncoder()` | creates one column per category |
> | `sparse_output=False` | returns a normal array. The default saves memory by storing only the non-zero entries, which we cannot read directly. |

**Multicollinearity, and dropping one column**

One-hot encoding has a side effect. The columns it creates are redundant: every row holds exactly
one 1, so the columns always add up to 1. That means you can work out any column from the others.
If a house is not in Edwards, not in Gilbert and not in NAmes, then it must be in Crawfor.

When one column can be computed from the others, the columns are **collinear**. When several can,
we call it **multicollinearity**. It is a problem for linear models: there is no longer a single
best set of coefficients, so the values the model reports become unstable and hard to interpret.
Small changes in the data can produce very different coefficients.

The fix is to drop one column. You lose nothing, because that column carried no information the
others did not already give you. The dropped category becomes the **reference**: a row belongs to
it when every remaining column is 0.

The cell below encodes `neighborhood` again, this time dropping the first category.

In [16]:
reduced = OneHotEncoder(sparse_output=False, drop="first").set_output(transform="pandas")
reduced.fit_transform(homes[["neighborhood"]])

,neighborhood_Edwards,neighborhood_Gilbert,neighborhood_NAmes
0,0.0,1.0,0.0
1,1.0,0.0,0.0
2,0.0,1.0,0.0
3,0.0,0.0,0.0
4,0.0,0.0,1.0
5,1.0,0.0,0.0


`Crawfor` no longer has a column. Row 3 is the Crawfor house, and it now
holds 0 in all three columns. Four columns became three, and no information was lost.

Use `drop="first"` when the model is sensitive to collinear columns, which mainly means linear
models. Tree-based models, which we meet in later sessions, are not affected, and dropping a column
there only makes the output harder to read.

> **In `scikit-learn`**
>
> | Code | What it does |
> |---|---|
> | `drop="first"` | removes the first category of every column |
> | `drop="if_binary"` | removes one column only for variables with exactly two categories, which is the common middle ground |

**High cardinality.**

*Cardinality* means the number of distinct values in a column.

One-hot
encoding a column with 5,000 distinct values produces 5,000 columns, which you cannot use. One
solution needs no new tools: group the rare categories into a single "infrequent" category, and
keep a separate column only for values that appear often enough to learn from. The cell below keeps
categories that appear at least twice.

In [17]:
pooled = OneHotEncoder(sparse_output=False, min_frequency=2).set_output(transform="pandas")
pooled.fit_transform(homes[["neighborhood"]])

,neighborhood_Edwards,neighborhood_Gilbert,neighborhood_infrequent_sklearn
0,0.0,1.0,0.0
1,1.0,0.0,0.0
2,0.0,1.0,0.0
3,0.0,0.0,1.0
4,0.0,0.0,1.0
5,1.0,0.0,0.0


`Gilbert` and `Edwards` each appear twice, so they keep their own columns.
`Crawfor` and `NAmes` appear once each, so they are grouped into
`neighborhood_infrequent_sklearn`. Four columns become three. More importantly, no column is based
on just one row.

> **In `scikit-learn`**
>
> | Code | What it does |
> |---|---|
> | `min_frequency=2` | categories appearing fewer than twice are grouped into a single `infrequent` column |
> | `handle_unknown="ignore"` | if a category was not seen during `fit`, produce a row of zeros instead of raising an error. Part 5 shows why this matters. |

### 2.3.2. Ordinal variables

**Ordinal encoding** is for **ordered** variables. Each category maps to 0, 1, 2, and so on,
following its rank. You get one column instead of many, and the order is kept.

There is a danger here. You must *give* the encoder the order. If you do not, it sorts the values
alphabetically, and for quality grades that result has no meaning. Our grades run `Po` < `Fa` <
`TA` < `Gd` < `Ex`, that is Poor < Fair < Typical < Good < Excellent. The next cell encodes them
both ways.

In [18]:
QUALITY_ORDER = ["Po", "Fa", "TA", "Gd", "Ex"]

correct = OrdinalEncoder(categories=[QUALITY_ORDER]).fit_transform(homes[["quality"]])
alphabetical = OrdinalEncoder().fit_transform(homes[["quality"]])

pd.DataFrame({
    "quality": homes["quality"],
    "correct order": correct.ravel(),
    "alphabetical (wrong)": alphabetical.ravel(),
})

,quality,correct order,alphabetical (wrong)
0,Gd,3.0,2.0
1,TA,2.0,3.0
2,Gd,3.0,2.0
3,Ex,4.0,0.0
4,TA,2.0,3.0
5,Fa,1.0,1.0


When we give the order, we get `Fa` → 1, `TA` → 2, `Gd` → 3, `Ex` → 4. The
rank is kept.

When we do not give the order, the encoder sorts alphabetically: `Ex` < `Fa` < `Gd` < `TA`. It then
produces `Ex` → 0, `Fa` → 1, `Gd` → 2, `TA` → 3. **Excellent is now the lowest grade and Typical
is the highest.** The code runs, the output looks normal, and every result after this point is
wrong. This is the most common silent error in categorical encoding.

> **In `scikit-learn`**
>
> | Code | What it does |
> |---|---|
> | `OrdinalEncoder(categories=[...])` | maps categories to 0, 1, 2 … **in the order you give**. Without this argument it sorts alphabetically, which is the error shown above. |
> | `handle_unknown="use_encoded_value"` | do not raise an error when a category was not seen during `fit` |
> | `unknown_value=-1` | the number to use for such a category |

One more warning, even when the order is correct. Mapping to 0, 1, 2, 3, 4 says that the steps are
**equal**: that the distance from Fair to Typical is the same as the distance from Good to
Excellent. Often it is not. One-hot encoding gives each level its own column, so the model can
learn the real distances instead of assuming them.

## 2.4 Creating new features

Sometimes the quantity you need is not in the data, but you can compute it. A lot area and a living
area are each useful. The **share of the lot that is built on** may be more useful, and a model
cannot produce it on its own: a ratio is not something you can build by adding columns together.

The cell below creates that feature with a `FunctionTransformer`, so it is produced by a pipeline
step and not by editing the DataFrame.

> 🛠️ **In `scikit-learn`**, `FunctionTransformer()` converts a function into a transformer.
> The function then gains the `fit()`, `transform()` and `fit_transform()` methods, so it can be
> used inside a pipeline like any other step.
>
> | Code | What it does |
> |---|---|
> | `FunctionTransformer(func)` | wraps `func` so it becomes a transformer |
> | `feature_names_out="one-to-one"` | tells it the output columns keep the input names |

In [19]:
def add_built_fraction(frame: pd.DataFrame) -> pd.DataFrame:
    """Add the share of the lot occupied by living area.

    Args:
        frame: A frame containing ``living_area`` and ``lot_area``.

    Returns:
        A copy of the frame with a ``built_fraction`` column appended.
    """
    frame = frame.copy()
    frame["built_fraction"] = frame["living_area"] / frame["lot_area"]
    return frame


FunctionTransformer(add_built_fraction).fit_transform(homes[["lot_area", "living_area"]]).round(4)

,lot_area,living_area,built_fraction
0,8450,1710,0.2024
1,9600,1262,0.1315
2,11250,1786,0.1588
3,9550,1717,0.1798
4,92000,2198,0.0239
5,7420,1362,0.1836


The outlier house is now unusual for a second reason. Its `built_fraction` is
**0.024**, while the others are between 0.13 and 0.20. It is a large lot with an ordinary house on
it. That is a more useful description than "large number in the lot column".

Notice how this feature is computed. It uses only values from **one row**, so it needs no
information from any other house. That property matters in Part 5.

## 2.5 Outliers

An **outlier** is a value that is far from the rest of the distribution. In `lot_area`, row 4 holds
92,000 square feet while the other rows are near 9,000.

First decide what kind of outlier it is, because the word covers two different cases. It may be an
**error**: a typing mistake, a broken sensor, or a placeholder such as 9999. So in this case,
you should correct it or remove it. Or it may be **real but rare**: one very large property among
smaller ones. Then it is true, and removing it removes information.

One common rule for *finding* candidates is the **interquartile range**:

$$\text{IQR} = Q_3 - Q_1, \qquad \text{fence} = [\,Q_1 - 1.5 \times \text{IQR},\ Q_3 + 1.5 \times \text{IQR}\,]$$

where:

- $Q_1$ — the first quartile, the value below which 25% of the data falls
- $Q_3$ — the third quartile, the value below which 75% of the data falls
- $\text{IQR}$ — the spread of the middle half of the data; always $\geq 0$
- the **fence** is the range of normal values. Values outside it are flagged for you to check.

The cell below applies this rule to `lot_area`. It uses the `pandas` method `.quantile()` to get
$Q_1$ and $Q_3$.

In [20]:
q1, q3 = homes["lot_area"].quantile([0.25, 0.75])
print(f"Q1 = {q1}")
print(f"Q2 = {q3}")

Q1 = 8725.0
Q2 = 10837.5


In [21]:
iqr = q3 - q1
lower_limit = q1 - 1.5 * iqr
upper_limit = q3 + 1.5 * iqr
print(f"IQR = {iqr}")
print(f"fence = [{lower_limit}, {upper_limit}]")

IQR = 2112.5
fence = [5556.25, 14006.25]


In [22]:
print("Normal values:")
homes.query("@lower_limit < lot_area < @upper_limit").lot_area.to_list()

Normal values:


[8450, 9600, 11250, 9550, 7420]

The fence runs from **5,556 to 14,006**. The value **92,000** falls outside
it.

Once you have found an outlier, there are three common responses:

- **Clip.** Move extreme values back to the fence. The row stays, but its influence is limited.
- **Transform.** Apply a logarithm, which compresses the long tail.
- **Scale robustly.** Centre and scale with the median and the IQR. An outlier moves these very
  little. The mean and the standard deviation, in contrast, move a lot.

The cell below applies all three so you can compare them:

> 🛠️ **In `scikit-learn`**, `RobustScaler()` is the third scaling method promised in section 2.1.
> It works like `StandardScaler`, but it centres on the **median** instead of the mean, and divides
> by the **IQR** instead of the standard deviation. Both of those are barely moved by an outlier,
> which is the whole point.

The cell below applies all three responses to `lot_area` so you can compare them.

In [23]:
clipper = FunctionTransformer(lambda X: X.clip(lower=lower_limit, upper=upper_limit))
logger = FunctionTransformer(np.log1p)

comparison = pd.DataFrame({
    "original": homes["lot_area"],
    "clipped": clipper.fit_transform(homes[["lot_area"]])["lot_area"],
    "log1p": logger.fit_transform(homes[["lot_area"]])["lot_area"],
    "StandardScaler": StandardScaler().fit_transform(homes[["lot_area"]]).ravel(),
    "RobustScaler": RobustScaler().fit_transform(homes[["lot_area"]]).ravel(),
})
comparison.round(2)

,original,clipped,log1p,StandardScaler,RobustScaler
0,8450,8450.00,9.04,-0.47,-0.53
1,9600,9600.00,9.17,-0.44,0.01
2,11250,11250.00,9.33,-0.38,0.79
3,9550,9550.00,9.16,-0.44,-0.01
4,92000,14006.25,11.43,2.23,39.02
5,7420,7420.00,8.91,-0.51,-1.02


Compare the last two columns with care. They behave in opposite ways.

**`StandardScaler`** divides by the standard deviation, and the outlier makes that value large. The
five ordinary houses are pushed into the narrow range −0.51 to −0.38, where you can barely tell
them apart. The outlier sits at 2.23. One extreme value has removed the detail from every other
row.

**`RobustScaler`** divides by the IQR, which the outlier barely changes. The five ordinary houses
now spread from −1.02 to 0.79, and you can read the differences between them. The outlier lands at
**39.02**, far from the rest.

Neither result is correct in general. They serve different needs. `StandardScaler` keeps the
outlier close to the other values, but it compresses those values. `RobustScaler` keeps the detail
in the main group, and leaves the outlier clearly far away.

`log1p` compresses the tail without removing anything. The value 92,000 becomes 11.4, and 8,450
becomes 9.0. The large lot is still the largest, but it is no longer forty times larger. On this
six-row table the skew barely changes (2.44 to 2.33), because six points cannot describe a
distribution. On the real data in Part 4 the change is large.

> **In `scikit-learn`**
>
> | Code | What it does |
> |---|---|
> | `Series.quantile([0.25, 0.75])` | the two quartiles the IQR rule needs. This one is `pandas`, not `scikit-learn`. |
> | `np.log1p(x)` | computes $\log(1 + x)$. The "1 +" means a value of 0 stays 0, instead of causing an error. |
> | `RobustScaler()` | centres on the median and scales by the IQR |
> | `FunctionTransformer(func)` | turns any function into a transformer, so it can go inside a pipeline. This is how `clip` and `log1p` above became pipeline steps instead of loose `pandas` calls. |
> | `feature_names_out="one-to-one"` | tells `FunctionTransformer` that the output columns keep the input names |

# Part 3 — A diagnostic before you engineer

Parts 1 and 2 gave you a catalogue of techniques. When you meet a new dataset, the question is
*which technique does each column need*. You can answer it by inspection, before any modelling.

For each column, ask six questions:

| Question | What the answer decides |
|---|---|
| Is it numeric or categorical? | which half of the catalogue applies |
| If categorical, is it ordered? | one-hot encoding or ordinal encoding |
| How many values are missing? | whether to impute, delete, or flag |
| What does the absence *mean*? | which imputation strategy is correct |
| How many distinct values? | the cost of one-hot encoding; whether to group rare categories |
| How skewed is it, and how many outliers? | whether to transform, clip, or scale robustly |

You can compute the answer to five of these six questions. The one you cannot compute is knowing
what a missing value means. For this, you need a data dictionary or someone who knows the subject.
Keep this in mind: **the one question a script cannot answer is the most important one.**

The cell below builds a reusable diagnostic that answers the five computable questions.

In [24]:
homes.select_dtypes(exclude="str").columns

Index(['lot_area', 'living_area', 'frontage', 'bedrooms'], dtype='str')

In [25]:
def diagnose(df: pd.DataFrame) -> pd.DataFrame:
    """Report the properties of each column that determine its preparation.

    Args:
        df: The raw feature DataFrame to inspect.

    Returns:
        One row per column with type, missingness, cardinality, skew and
        outlier count.
    """
    report = []
    num_cols = df.select_dtypes(exclude="str").columns
    for name in df.columns:
        column = df[name]
        numeric = pd.api.types.is_numeric_dtype(column)
        entry = {
            "column": name,
            "kind": "numeric" if numeric else "categorical",
            "missing": int(column.isna().sum()),
            "missing %": round(column.isna().mean() * 100, 1),
            "distinct": int(column.nunique()),
        }
        if name in num_cols:
            q1_, q3_ = column.quantile([0.25, 0.75])
            iqr_ = q3_ - q1_
            outside = (column < q1_ - 1.5 * iqr_) | (column > q3_ + 1.5 * iqr_)
            entry["skew"] = round(column.skew(), 2)
            entry["outliers"] = int(outside.sum())
            entry["range"] = f"{column.min():,.0f} - {column.max():,.0f}"
        else:
            entry["skew"] = np.nan
            entry["outliers"] = np.nan
            entry["range"] = ""
        report.append(entry)
    return pd.DataFrame(report)


diagnose(homes)

,column,kind,missing,missing %,distinct,skew,outliers,range
0,lot_area,numeric,0,0.0,6,2.44,1.0,"7,420 - 92,000"
1,living_area,numeric,0,0.0,6,0.38,0.0,"1,262 - 2,198"
2,frontage,numeric,2,33.3,4,-0.06,0.0,60 - 84
3,bedrooms,numeric,0,0.0,3,-0.31,0.0,2 - 4
4,quality,categorical,0,0.0,4,NaN,NaN,
5,neighborhood,categorical,0,0.0,4,NaN,NaN,
6,garage,categorical,2,33.3,2,NaN,NaN,


On the toy table the report confirms what we already knew. That is why we try
it here first. `frontage` and `garage` each have two gaps. `lot_area` has one outlier and a skew of
2.44. `neighborhood` has four distinct values and `quality` has three.

To turn a report like this into a plan, use a small set of rules:

| What you see | What to do |
|---|---|
| categorical, few distinct values | one-hot encode |
| categorical, **ordered** values | ordinal encode, and give the order yourself |
| categorical, many distinct values | group rare categories with `min_frequency` |
| missing, and the value exists | impute with the median or mean; consider a missing indicator |
| missing, and the thing does not exist | impute with a **constant** that names the absence |
| high skew, no zeros | apply `log1p` |
| outliers present | clip, or scale robustly |
| large range compared with other columns | scale |

Notice again what the table cannot tell you. It cannot say whether the blanks in `garage` mean *no
garage* or *not recorded*. It only reports that two values are missing. The data dictionary tells
you what that means.

# Part 4 — Applying it to the Ames housing dataset

## 4.1 The data

The **Ames housing dataset** holds records of house sales in Ames, Iowa, between 2006 and 2010. The
city assessor's office collected it.

- **1,460 rows.** One row is one completed sale of one house.
- **79 feature columns** in the full file. They describe size, quality, location, age and sale
  conditions.
- **`SalePrice`** is the target: the price the house sold for, in US dollars. We set it aside,
  because this session prepares features and does not predict.

We use **fourteen** of those columns, chosen to cover every problem in the catalogue. Their names
are abbreviated in the original file, so here is what each one means:

| Column | Meaning |
|---|---|
| `GrLivArea` | above-ground living area, in square feet |
| `TotalBsmtSF` | total basement area, in square feet |
| `LotArea` | size of the whole plot, in square feet |
| `LotFrontage` | length of street touching the plot, in feet |
| `MasVnrArea` | area of masonry veneer (a decorative brick or stone surface), in square feet |
| `YearBuilt` | year the house was built |
| `GarageCars` | how many cars fit in the garage |
| `OverallQual` | overall quality of materials and finish, graded 1 to 10 |
| `KitchenQual` | kitchen quality, graded `Po` to `Ex` |
| `ExterQual` | quality of the exterior material, graded `Po` to `Ex` |
| `HeatingQC` | heating quality and condition, graded `Po` to `Ex` |
| `FireplaceQu` | fireplace quality, graded `Po` to `Ex`; missing when there is no fireplace |
| `Neighborhood` | which part of Ames the house is in |
| `GarageType` | where the garage is, for example `Attchd` or `Detchd` |

The four columns graded `Po` to `Ex` use the same scale as the toy table: Poor, Fair, Typical, Good,
Excellent.

The cell below loads these columns and runs the diagnostic.

In [26]:
NUMERIC = ["GrLivArea", "TotalBsmtSF", "LotArea", "LotFrontage",
           "MasVnrArea", "YearBuilt", "GarageCars", "OverallQual"]
ORDERED = ["KitchenQual", "ExterQual", "HeatingQC", "FireplaceQu"]
NOMINAL = ["Neighborhood", "GarageType"]
COLUMNS = NUMERIC + ORDERED + NOMINAL

houses = pd.read_csv("data/ames_housing.csv")
X = houses[COLUMNS]

diagnose(X)

FileNotFoundError: [Errno 2] No such file or directory: 'data/ames_housing.csv'

Read the report, and the plan follows from it.

**Missing values, three kinds.** `FireplaceQu` is missing in **47.3%** of the rows. Those houses
have no fireplace, so the blank means *does not apply*, and it needs a named category. `GarageType`
at 5.5% is the same case. `LotFrontage` at 17.7% is different: every house has street frontage, so
the value is *unknown*, and it needs a median. `MasVnrArea` at 0.5% means *no veneer*, so it needs
a zero.

**Skew and outliers.** `LotArea` has a skew of **12.21** and 69 flagged values, and it ranges from
1,300 to 215,245. It is a clear candidate for `log1p`. `MasVnrArea` (2.67) and `LotFrontage` (2.16)
are also skewed. `YearBuilt` has a skew of −0.61, which is small and needs no action.

**Cardinality.** `Neighborhood` has 25 distinct values, so grouping the rare ones is worth
considering. The four quality columns have 4 or 5 values each, and they are **ordered**. The report
cannot tell us that. The data dictionary can: it records them with the same codes as our toy
table, `Po` < `Fa` < `TA` < `Gd` < `Ex`.

**Scale.** `LotArea` reaches 215,245, while `OverallQual` runs from 1 to 10 and `GarageCars` from 0
to 4. We need to scale.

## 4.2 Building a preprocessing pipeline

We will explore the fundamentals of building preprocessing **pipelines** using Scikit-Learn. Pipelines are essential for creating efficient, reusable, and organized workflows for data preparation, especially when handling datasets with a mix of numerical and categorical features.

First, let's address the traditional approach centered around ``Pipeline``, which serve as the backbone of Scikit-Learn's pipeline implementations. In this approach, transformers are sequentially applied to all features.

The cell below will try to chain two steps to preprocess the Ames dataset:

In [ ]:
naive_pipeline = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
])
naive_pipeline

In [ ]:
try:
    naive_pipeline.fit_transform(X)
except Exception as error:
    print(f"{type(error).__name__}: {error}")

It fails, and the message is clear: `Cannot use median strategy with
non-numeric data: could not convert string to float: 'Gd'`.

Basically, the pipeline is breaking because it is trying to obtain the median of non-numerical columns. The median of `Gd` and `TA` has no meaning. The lesson is that **one sequence of steps cannot serve
every column**. Each family of columns needs its own treatment. We must apply those treatments
separately and then join the results.

`ColumnTransformer` does this.

> **In `scikit-learn`**
>
> | Code | What it does |
> |---|---|
> | `Pipeline([...])` | runs a list of steps in order. It is itself a transformer, so it can be used anywhere one is expected. |
> | `("name", transformer)` | one step of a `Pipeline`: a label you choose, and the transformer to run |
> | `ColumnTransformer([...])` | applies a different transformer to each group of columns, then joins the results side by side |
> | `("name", transformer, columns)` | one group of a `ColumnTransformer`: a label, the transformer, and which columns it receives |
> | `verbose_feature_names_out=False` | keeps the output column names short, without the group label in front |

To address the limitations encountered earlier, we can use the `ColumnTransformer`. This tool allows us to apply different transformations to subsets of features (e.g., numerical and categorical columns) independently, then concatenate the results into a single unified dataset.

Using `ColumnTransformer` is essential when dealing with mixed data types, as it ensures that each feature is handled with the appropriate transformations.

The cell below builds one branch pipeline for each family. Numeric columns are imputed and scaled. Ordered
categories are imputed and ordinal-encoded. Nominal categories are imputed and one-hot encoded.

In [ ]:
QUALITY_SCALE = ["NA", "Po", "Fa", "TA", "Gd", "Ex"]

numeric_branch = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
])
ordered_branch = Pipeline([
    ("impute", SimpleImputer(strategy="constant", fill_value="NA")),
    ("encode", OrdinalEncoder(categories=[QUALITY_SCALE] * len(ORDERED),
                              handle_unknown="use_encoded_value", unknown_value=-1)),
])
nominal_branch = Pipeline([
    ("impute", SimpleImputer(strategy="constant", fill_value="None")),
    ("encode", OneHotEncoder(sparse_output=False, handle_unknown="ignore", min_frequency=10)),
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_branch, NUMERIC),
    ("ordered", ordered_branch, ORDERED),
    ("nominal", nominal_branch, NOMINAL),
], verbose_feature_names_out=False).set_output(transform="pandas")
preprocessor

`OrdinalEncoder` needs one list of categories per column it receives. The cell below shows what `[QUALITY_SCALE] * len(ORDERED)` produces: the same scale, repeated once for each of the four quality columns.

In [ ]:
[QUALITY_SCALE] * len(ORDERED)

In [ ]:
prepared = preprocessor.fit_transform(X)

print(f"Initial number of columns: {X.shape[1]}")
print(f"Final number of columns: {prepared.shape[1]}")
prepared.head()

It runs. Fourteen columns became **42 features**. The eight numeric columns
passed through and were scaled. The four quality columns became one column each. The two nominal
columns produced the rest.

Look at `FireplaceQu` in the output. Its 690 missing values became the category `NA`. That code is
our own addition, not part of the dataset: we chose it to mean "there is no fireplace". We placed
it at position 0 in `QUALITY_SCALE`, *below* `Po`. This is deliberate and correct: having no
fireplace ranks below having a poor one.

Two items from the diagnostic are still open: the skew of `LotArea`, and the blanks in `MasVnrArea`
that mean zero rather than median. The cell below adds a fourth branch for the skewed columns.

In [ ]:
SKEWED = ["LotArea", "MasVnrArea", "LotFrontage", "TotalBsmtSF"]
PLAIN = [c for c in NUMERIC if c not in SKEWED]

skewed_branch = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("log", FunctionTransformer(np.log1p, feature_names_out="one-to-one")),
    ("scale", StandardScaler()),
])

preprocessor = ColumnTransformer([
    ("skewed", skewed_branch, SKEWED),
    ("numeric", numeric_branch, PLAIN),
    ("ordered", ordered_branch, ORDERED),
    ("nominal", nominal_branch, NOMINAL),
], verbose_feature_names_out=False).set_output(transform="pandas")
preprocessor

In [ ]:
prepared = preprocessor.fit_transform(X)
prepared

In [ ]:
print("skew before and after log1p:")
print(X[SKEWED].skew())

In [ ]:
print("skew after log1p:")
print(prepared[SKEWED].skew())

Three of the four columns improve. One gets much worse:

| Column | Skew before | After `log1p` |
|---|---|---|
| `LotArea` | 12.21 | **−0.14** |
| `LotFrontage` | 2.16 | **−0.87** |
| `MasVnrArea` | 2.67 | **0.50** |
| `TotalBsmtSF` | 1.52 | **−5.15** |

`TotalBsmtSF` was slightly skewed to the right. After the transform it is strongly skewed to the
left. The reason is worth understanding, because it is not simply "the column contains zeros".
`MasVnrArea` is more than half zeros, and it improved.

The difference is **where most of the data sits**. Most houses have a basement of 800 to 1,500
square feet, and `log1p` compresses that whole range into a narrow band around 7. But 37 houses
have no basement, and `log1p` leaves them at exactly 0. Those 37 values are now far below a tight
group, which creates a long tail on the left. `MasVnrArea` avoids this because its non-zero values
are spread widely, so compressing them does not produce a tight group.

**A log transform assumes a positive quantity.** When zero means "this does not exist", the column
is really two things at once: a yes/no answer and an amount. Such a column is better served by an
indicator flag next to the raw value than by a transform that treats the zeros as very small
amounts.

The report could not tell us this. It flagged all four columns in the same way. Only by asking
*why* each column was skewed did we find that the standard treatment was wrong for one of them.

# Part 5 — Why this belongs in a pipeline: leakage

Every transformer above **learned something** before it could transform anything. The imputer
learned a median. The scaler learned a mean and a standard deviation. The encoder learned which
categories exist.

That learning is the whole problem. If a transformer learns from data that you plan to use for
evaluation, then information from that data has entered your preparation. This is called **data
leakage**, and it means your evaluation measures something that will not be true in production.

We do not need model scores to show this. We can look directly at the numbers the transformers
learn. The cell below splits the data, then fits the same imputer twice: once on all the rows, and
once on the training rows only.

> **In `scikit-learn`**
>
> | Code | What it does |
> |---|---|
> | `train_test_split(X, test_size=0.2)` | splits the rows at random, here keeping 20% aside for testing |
> | `random_state=42` | fixes the random choice, so the split is the same every time you run the notebook |

In [ ]:
X_train, X_test = train_test_split(X, test_size=0.2, random_state=RANDOM_STATE)

fitted_on_all = SimpleImputer(strategy="median").fit(X[["LotFrontage"]])
fitted_on_train = SimpleImputer(strategy="median").fit(X_train[["LotFrontage"]])

print(f"LotFrontage median learned from ALL rows   : {fitted_on_all.statistics_[0]}")
print(f"LotFrontage median learned from TRAIN only : {fitted_on_train.statistics_[0]}")

**69.0 against 70.0.** The two numbers are different, so different values go
into the 259 gaps. The first number was computed using rows that we are supposed to hold back.

Scaling shows the problem more clearly, because a scaler learns two values and applies them to
every row. The next cell compares what each version does to the same three test houses.

In [ ]:
scaler_all = StandardScaler().fit(X[["LotArea"]])
scaler_train = StandardScaler().fit(X_train[["LotArea"]])

print("mean/scale from ALL rows:") 
print(f"{scaler_all.mean_.round(2)} / {scaler_all.scale_.round(2)}")

print("mean/scale from TRAIN only")
print(f"{scaler_train.mean_.round(2)} / {scaler_train.scale_.round(2)}")

In [ ]:
three_houses = X_test[["LotArea"]].head(3)
pd.DataFrame({
    "raw LotArea": three_houses["LotArea"].values,
    "scaled (fitted on all)": scaler_all.transform(three_houses).ravel().round(4),
    "scaled (fitted on train)": scaler_train.transform(three_houses).ravel().round(4),
})

The learned values are different. The mean is **10,516.8** against
**10,689.6**, and the spread is **9,977.8** against **10,754.8**. So every scaled value is
different. The same house is given to the model as a different number, depending on which rows the
scaler was allowed to see.

The differences here are small, in the third decimal place. Do not let that reassure you, for two
reasons. First, with 1,460 rows the two subsets really are similar. Second, small differences are
exactly what makes leakage hard to notice. The size of the difference is not the problem. The
problem is that your test set no longer measures anything honestly.

Encoders learn as well, and they fail more loudly. The cell below fits an encoder on four toy
houses, then applies it to two houses that contain a neighbourhood it has never seen.

In [ ]:
seen = homes[["neighborhood"]].head(3)
seen

In [ ]:
unseen = homes[["neighborhood"]].tail(3)
unseen

In [ ]:
strict = OneHotEncoder(sparse_output=False).fit(seen)

try:
    strict.transform(unseen)
except Exception as error:
    print(f"{type(error).__name__}: {error}")

In [ ]:
tolerant = OneHotEncoder(sparse_output=False, handle_unknown="ignore").set_output(transform="pandas")
tolerant.fit(seen)
tolerant.transform(unseen)

By default the encoder **raises an error**, because `NAmes` was not in the
categories it learned. With `handle_unknown="ignore"` it produces a row of zeros instead. That is
the honest answer: "this row belongs to none of the categories I know".

If you had one-hot encoded the whole dataset in `pandas` before the split, you would never have
seen this. `NAmes` would have had its own column, and the failure would have appeared later, in
production, against a neighbourhood the training data never contained.

**This is the argument for pipelines**, and it is not about tidy code:

- A `Pipeline` calls `fit` on its steps only inside its own `.fit()`. At prediction time it calls
  `transform` only. "Fit on training data, transform everything" stops being a rule you must
  remember, and becomes what the object does.
- Everything the pipeline learned — medians, means, category lists — is stored in one fitted object
  that you can save.
- The same work in `pandas`, such as `df.fillna(df.median())` or `pd.get_dummies(df)`, computes
  those values from whatever table is in front of you. Before a split, that table is the whole
  dataset. Every such line is a leak.

The techniques in Part 2 are the same in both cases. The difference is that a pipeline remembers
what it learned, and which rows it learned from.

# Part 6 — Wrap-up

**Classify each column before you transform it.** Numeric or categorical; ordered or not. The
nominal/ordinal distinction is invisible in the data type, and it decides your encoding.

**Ask what a blank means** before you choose how to fill it. "Unknown", "does not apply" and "is
zero" are three different cases with three different correct answers, and no script can tell them
apart.

**An outlier is a value to check, not a value to delete.** Find candidates with the IQR rule. Then
decide whether each one is an error or a real rare value, and clip, transform or scale robustly
based on that.

**Diagnose first, then engineer.** The report in Part 3 answers five of the six questions
automatically. Run it on any dataset and most of the plan becomes clear.

**Every transformer is fitted, so every transformer can leak.** This is why preparation belongs in
a pipeline, and not in `pandas` calls written before the split.

**What we found in Ames:**

| Column | The diagnostic said | What we did |
|---|---|---|
| `FireplaceQu` | 47.3% missing | named category `NA`, ranked below `Po` |
| `GarageType` | 5.5% missing | named category `None` |
| `LotFrontage` | 17.7% missing, skew 2.16 | median, then `log1p` |
| `LotArea` | skew 12.21, 69 outliers | `log1p`; skew fell to −0.14 |
| `MasVnrArea` | skew 2.67 | `log1p` improved it to 0.50 |
| `TotalBsmtSF` | skew 1.52 | `log1p` made it **much worse** (−5.15); 37 houses have no basement |
| quality columns | 4 or 5 values, ordered | ordinal encoding with the order given |
| `Neighborhood` | 25 values | one-hot encoding, rare categories grouped |

Fourteen columns became 42 features. One of the diagnostic's recommendations turned out to be wrong
when we looked at it closely. That is why the report is a starting point and not an answer.

### What is next

We prepared these features without asking whether the preparation **helped**. To answer that we
need an honest way to measure, and one score on one split is not enough. **Session 3** covers
cross-validation and model selection. It also returns to several choices we made here, to find out
which of them were worth making.

## Exercises

1. **Get the order wrong on purpose.** Build the ordered branch again, this time without the
   `categories` argument, so the encoder sorts alphabetically. Print both encodings of
   `KitchenQual` side by side. Describe exactly what a model would now believe about kitchen
   quality.

2. **Treat the basement correctly.** Part 4 showed that `log1p` damages `TotalBsmtSF`, because 37
   houses have no basement. Build a branch that produces two features instead: a 0/1 flag for
   whether a basement exists, and the basement area. Compute the skew of the area using only the
   houses that have a basement. Does the transform behave better now?

3. **Group more categories.** Run the nominal branch again with `min_frequency` set to 5, then 25,
   then 100. How many `Neighborhood` columns survive each time? At what point would you say that
   too much information has been lost?

4. **Diagnose your own dataset.** Take any table you have, from another course, from work, or from
   a public source. Run `diagnose()` on it. Write the treatment plan for every column, and mark
   clearly which decisions came from the report and which needed the data dictionary. Then build
   the `ColumnTransformer` that carries out your plan.